# 01 — Data Exploration & Cleaning

**Objective:** Load, inspect, and validate the tradable asset universe used to replicate CSI MARP 930929.

This notebook is read-only with respect to the locked OOS period (2022-01-01 onward).
No parameter tuning uses data beyond 2021-12-31.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from src.conventions import *

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

In [ ]:
# Load clean data
prices = pd.read_parquet(DATA_CLEAN / 'prices.parquet')
returns = pd.read_parquet(DATA_CLEAN / 'returns.parquet')
marp_official = pd.read_parquet(DATA_CLEAN / 'marp_official.parquet')
factors = pd.read_parquet(DATA_FACTORS / 'china_ff3_proxy.parquet')

prices_oos = pd.read_parquet(DATA_OOS / 'prices_oos.parquet')
returns_oos = pd.read_parquet(DATA_OOS / 'returns_oos.parquet')

print(f'Full-sample prices : {prices.shape[0]:,} rows × {prices.shape[1]} cols')
print(f'  Date range       : {prices.index.min().date()} → {prices.index.max().date()}')
print(f'  Trading days     : {prices.shape[0]}')
print(f'OOS prices         : {prices_oos.shape[0]:,} rows')

In [ ]:
# Asset universe
asset_labels = pd.Series(ETF_UNIVERSE)
asset_labels

In [ ]:
# Coverage check — how many non-NA observations per asset?
coverage = pd.DataFrame({
    'Ticker': prices.columns,
    'Non-NaN': prices.notna().sum().values,
    'Pct Complete': (prices.notna().sum() / len(prices) * 100).values,
    'First Date': [prices[c].first_valid_index().date() for c in prices.columns],
    'Last Date':  [prices[c].last_valid_index().date() for c in prices.columns],
})
coverage

In [ ]:
# Price time series — normalised to 1 at first common date
common_start = prices.dropna().index[0]
norm_prices = prices / prices.loc[common_start]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#00BCD4', '#795548', '#607D8B']
for i, col in enumerate(norm_prices.columns):
    axes[0].plot(norm_prices.index, norm_prices[col], color=colors[i % len(colors)], 
                 linewidth=0.8, alpha=0.85, label=col)

axes[0].axvline(x=pd.Timestamp(OOS_START), color='black', linestyle='--', linewidth=1.2, alpha=0.7)
axes[0].annotate('OOS →', (pd.Timestamp(OOS_START), axes[0].get_ylim()[1]*0.95),
                 fontsize=10, ha='left', color='black')
axes[0].set_title('Normalised Prices (base = first common trading day)')
axes[0].legend(loc='upper left', fontsize=8, ncol=4)
axes[0].set_ylabel('Normalised Price')

# Subset: ETF universe only
etfs = ['510300', '510500', '511010', '518880', '159980']
etf_labels = ['CSI 300', 'CSI 500', '5Y Treasury', 'Gold', 'Commodity']
norm_etfs = prices[etfs] / prices[etfs].loc[common_start]
for i, col in enumerate(etfs):
    axes[1].plot(norm_etfs.index, norm_etfs[col], color=colors[i], linewidth=1.0, label=etf_labels[i])
axes[1].axvline(x=pd.Timestamp(OOS_START), color='black', linestyle='--', linewidth=1.2, alpha=0.7)
axes[1].set_title('Normalised ETF Prices (tradable universe)')
axes[1].legend(loc='upper left', fontsize=9)
axes[1].set_ylabel('Normalised Price')

fig.tight_layout()
plt.show()

In [ ]:
# Daily returns distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(etfs):
    ret = returns[col].dropna() * 100
    axes[0].hist(ret, bins=80, alpha=0.4, color=colors[i], label=etf_labels[i], density=True)
axes[0].set_title('Daily Return Distribution (%, full sample)')
axes[0].set_xlabel('Daily Return (%)')
axes[0].legend(fontsize=8)

# Boxplot
returns_pct = returns[etfs].dropna() * 100
returns_pct.columns = etf_labels
bp = axes[1].boxplot([returns_pct[c].dropna() for c in etf_labels], 
                      labels=etf_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors[:5]):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1].set_title('Daily Return Distribution (Boxplot)')
axes[1].set_ylabel('Daily Return (%)')
axes[1].tick_params(axis='x', rotation=30)

fig.tight_layout()
plt.show()

In [ ]:
# Annual summary statistics (in-sample only)
is_returns = returns.loc[:IS_END]

stats = pd.DataFrame({
    'Ann. Return (%)': (is_returns[etfs].mean() * 252 * 100).round(2),
    'Ann. Vol (%)': (is_returns[etfs].std() * np.sqrt(252) * 100).round(2),
    'Sharpe': ((is_returns[etfs].mean() * 252 - RISK_FREE_ANNUAL) / (is_returns[etfs].std() * np.sqrt(252))).round(3),
    'Skewness': is_returns[etfs].skew().round(3),
    'Kurtosis': is_returns[etfs].kurtosis().round(3),
    'Max DD (%)': ((is_returns[etfs].cummin() - is_returns[etfs].cummax()) / is_returns[etfs].cummax()).min().round(4) * 100,
})
stats.index = etf_labels
stats

In [ ]:
# Correlation matrix — in-sample
corr = is_returns[etfs].corr()
corr.index = etf_labels
corr.columns = etf_labels

fig, ax = plt.subplots(figsize=(7, 6))
cmap = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(corr, annot=True, fmt='.3f', cmap=cmap, vmin=-1, vmax=1,
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax.set_title('In-Sample Correlation Matrix (2017–2021)')
fig.tight_layout()
plt.show()

In [ ]:
# Rolling 60-day correlation between equity and gold
rolling_corr = returns['510300'].rolling(60).corr(returns['518880'])

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(rolling_corr.index, rolling_corr, color='#795548', linewidth=1.0)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax.axvline(x=pd.Timestamp(OOS_START), color='black', linestyle='--', linewidth=1.2, alpha=0.7)
ax.set_title('Rolling 60-Day Correlation: CSI 300 ETF vs Gold ETF')
ax.set_ylabel('Correlation')
ax.set_ylim(-1, 1)
ax.annotate('OOS →', (pd.Timestamp(OOS_START), 0.9), fontsize=10, ha='left')
fig.tight_layout()
plt.show()

In [ ]:
# Factor proxy inspection
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(factors.index, factors['mkt_rf'] * 100, color='#2196F3', linewidth=0.6, label='mkt_rf (CSI300 − rf)')
axes[0].plot(factors.index, factors['smb'] * 100, color='#FF9800', linewidth=0.6, label='smb proxy (CSI500 − CSI300)')
axes[0].set_title('Daily Factor Returns (%)')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Return (%)')

# Cumulative factor returns
factor_cum = (1 + factors[['mkt_rf', 'smb']].dropna()).cumprod()
axes[1].plot(factor_cum.index, factor_cum['mkt_rf'], color='#2196F3', linewidth=1.0, label='mkt_rf (cumulative)')
axes[1].plot(factor_cum.index, factor_cum['smb'], color='#FF9800', linewidth=1.0, label='smb proxy (cumulative)')
axes[1].axhline(y=1, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title('Cumulative Factor Returns')
axes[1].legend(fontsize=9)
axes[1].set_ylabel('Cumulative Return')

fig.tight_layout()
plt.show()

In [ ]:
# CSI MARP 930929 official index inspection
marp_full = pd.read_parquet(DATA_CLEAN / 'marp_official.parquet')
marp_date = pd.to_datetime(marp_full['日期'])
marp_close = marp_full['收盘'].astype(float)

marp_price = pd.Series(marp_close.values, index=marp_date).sort_index()
marp_ret = marp_price.pct_change().dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(marp_price.index, marp_price, color='black', linewidth=1.2)
axes[0].set_title('CSI MARP 930929 — Official Index Level')
axes[0].set_ylabel('Index Level')

axes[1].plot(marp_ret.index, marp_ret * 100, color='#607D8B', linewidth=0.5, alpha=0.7)
axes[1].set_title('CSI MARP 930929 — Daily Returns (%)')
axes[1].set_ylabel('Daily Return (%)')

print(f'CSI MARP annualised return: {marp_ret.mean()*252:.4f} ({marp_ret.mean()*252*100:.2f}%)')
print(f'CSI MARP annualised vol:    {marp_ret.std()*np.sqrt(252):.4f} ({marp_ret.std()*np.sqrt(252)*100:.2f}%)')
print(f'CSI MARP Sharpe (rf=2.2%):  {(marp_ret.mean()*252 - RISK_FREE_ANNUAL) / (marp_ret.std()*np.sqrt(252)):.3f}')

fig.tight_layout()
plt.show()

## Summary

- 8 assets in the panel, 5 tradable ETFs forming the core universe
- CSI 300 and CSI 500 provide equity exposure; 5Y Treasury for bonds; Gold for commodities/hedge
- Commodity ETF (159980) has limited history — need to handle in backtest
- CSI MARP 930929 serves as the non-tradable benchmark we aim to replicate
- Factor proxies available: mkt_rf (CSI300 − rf), smb (CSI500 − CSI300) — hml unavailable
- OOS period (2022+) will be used exclusively for evaluation